# Week 2 · Notebook 02: Detrending and the Box Least Squares Search
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/02_detrend_and_bls.ipynb)

**Goal:** flatten the star's variability without destroying transits, then *find* the planet blind with
Box Least Squares (BLS). You'll build each step by hand first, then use the package version: understand it,
then automate it. **Time:** 5–7 hours over the week.

### 🎯 By the end of this notebook you can
1. explain why we detrend and build a sliding-median detrender yourself;
2. choose a detrending window and measure what a bad choice does to a transit;
3. explain BLS as "fold at every trial period and look for a box", and run it;
4. read a periodogram: the peak, its aliases, SDE and SNR;
5. run two vetting tests that catch eclipsing binaries pretending to be planets.

### 🧭 Roadmap
| § | Topic |
|---|---|
| 1 | Why detrend? |
| 2 | Detrending by hand, then with the package |
| 3 | Gaps and sigma clipping |
| 4 | The window-length trade-off (measured on fake data) |
| 5 | BLS intuition: folding at trial periods |
| 6 | BLS by hand, then the real thing |
| 7 | Reading the periodogram |
| 8 | Is it a planet? Vetting |
| 9 | Your target |

**Reading:**
- Lightkurve: [Identifying transiting planet signals](https://lightkurve.github.io/lightkurve/tutorials/3-science-examples/exoplanets-identifying-transiting-planet-signals.html)
- Hippke et al. 2019, *Wotan* ([arXiv:1906.00966](https://arxiv.org/abs/1906.00966)), Sections 1–2
- Kovács, Zucker & Mazeh 2002, the BLS paper ([arXiv:astro-ph/0206099](https://arxiv.org/abs/astro-ph/0206099)), Intro + Sec. 2
<details><summary><b>📎 How to read this notebook (click to open)</b></summary>

| Symbol | Meaning |
|---|---|
| 🔮 **Predict** | Before running the next cell, write down what you expect. Being wrong is how you learn. |
| 👀 **What to look for** | How to read the output or plot you just made. |
| ✏️ **Your turn** | An exercise. 🟢 everyone · 🟡 stretch · 🔴 challenge. |
| 💡 / ✅ | Click to reveal a hint or a worked solution. Try first, then peek. |
| ⚠️ **Pitfall** | A mistake that real researchers make. |
| 🧠 **Concept check** | Short quiz. Answers are hidden under each question. |
| `check(...)` | Self-check: prints ✅ when your exercise answer is right. |

Run cells **top to bottom** with **Shift+Enter**. If things get weird: *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).
</details>

In [ ]:
# ⚙️ SETUP: run this cell first, every time you open the notebook. Works in Google Colab and locally.
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})

def check(name, value, expected, rtol=0.02):
    """Self-check for the exercises. Prints ✅, ❌, or ⏳ (not attempted yet)."""
    if value is None:
        print(f"⏳ {name}: not attempted yet (replace None with your answer)")
        return
    try:
        ok = bool(np.allclose(value, expected, rtol=rtol))
    except Exception:
        ok = value == expected
    print(f"✅ {name}: correct!" if ok else f"❌ {name}: not quite (you gave {value!r}). Check the hint.")

print("ready, repo root:", ROOT)

In [ ]:
from transitpipe import io, detrend, search, plotting, inject, utils

# Load Kepler-10 Q3-4: from the copy Notebook 01 saved if it exists, otherwise download (and save)
CACHE = os.path.join(ROOT, "data", "kepler10_q3q4.csv")
if os.path.exists(CACHE):
    lc = io.from_csv(CACHE)
    print("loaded from", CACHE)
else:
    lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3, 4])
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    io.to_csv(lc, CACHE)
STAR = {"rstar": 1.065, "rstar_err": 0.009, "mstar": 0.910, "mstar_err": 0.021}
lc.meta.update(STAR)
print(len(lc), "points")

## 1. Why detrend?
🔮 **Predict:** the star's slow variations vs the planet's ~150 ppm dip: which is bigger, and by how much?

In [ ]:
variability_ppm = (np.percentile(lc.flux, 99) - np.percentile(lc.flux, 1)) * 1e6
print(f"stellar/instrumental variability: ~{variability_ppm:.0f} ppm peak to peak")
print(f"Kepler-10 b transit depth:        ~150 ppm")
plotting.plot_raw(lc, title="Kepler-10 raw PDCSAP: the transits are hidden under these wiggles"); plt.show()

👀 The slow wiggles are several times bigger than the transit. BLS looks for *box-shaped dips*; slow
wiggles confuse it (a falling stretch of light curve looks like the start of a long dip). So we estimate the
slow **trend** and divide it out, leaving a flat light curve where only short dips remain.

The trick: a transit is **short** (hours); stellar variability is **slow** (days). A smoother with a window
*longer than the transit but shorter than the variability* separates them.

## 2. Detrending by hand: a sliding median
For each point, take the **median** of all points within ±½ window of it; call that the trend; divide it out.
(Why the median and not the mean? Remember Notebook 00: the median ignores a few low points, such as a transit.)

In [ ]:
import time as _time

def my_median_detrend(time, flux, window_days):
    trend = np.empty_like(flux)
    for i in range(len(time)):                              # one loop step per point (slow, but clear)
        near = np.abs(time - time[i]) < window_days / 2      # boolean mask: neighbours of point i
        trend[i] = np.median(flux[near])                     # robust local level
    return flux / trend, trend

tic = _time.time()
my_flat, my_trend = my_median_detrend(lc.time, lc.flux, 0.5)
print(f"hand-made detrending took {_time.time() - tic:.1f} s")

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(lc.time, lc.flux, ".", ms=1, color="k"); ax[0].plot(lc.time, my_trend, "C1", label="trend")
ax[1].plot(lc.time, my_flat, ".", ms=1, color="k")
ax[0].set_ylabel("raw flux"); ax[0].legend(); ax[1].set_ylabel("flattened"); ax[1].set_xlabel("Time [BKJD]")
plt.tight_layout(); plt.show()

The package version, `detrend.flatten`, does the same thing faster and handles **gaps** properly.
Let's check that the two agree (they should, except near the gap between quarters):

In [ ]:
flat_unclipped, trend = detrend.flatten(lc, method="median", window_days=0.5, return_trend=True)
diff = np.abs(flat_unclipped.flux - my_flat)
print(f"median difference vs your version: {np.median(diff) * 1e6:.2f} ppm, max: {diff.max() * 1e6:.0f} ppm")

## 3. Gaps and sigma clipping
**Gaps.** `flatten` cuts the light curve at every gap longer than `gap_days` and smooths each piece separately,
so the trend never "jumps" across missing data.

In [ ]:
segments = detrend.split_at_gaps(lc.time, gap_days=0.5)
for s in segments:
    print(f"segment {lc.time[s][0]:7.2f} -> {lc.time[s][-1]:7.2f} BKJD, {len(lc.time[s]):5d} points")

**Sigma clipping** removes outliers more than *n* robust standard deviations from 1.

🧠 Ours clips **only upward** outliers by default. Why?
<details><summary>Answer</summary>Transits are <i>low</i> points. A symmetric clip at 3–4σ can chop off the bottom of a deep transit, or the deepest points of a shallow one. Upward outliers (cosmic rays, flares) are never planets.</details>

In [ ]:
flat = detrend.sigma_clip(flat_unclipped, sigma=4)
removed = ~np.isin(flat_unclipped.time, flat.time)
print(f"clipped {removed.sum()} points")
print(f"noise after detrending: {detrend.estimate_noise_ppm(flat):.0f} ppm "
      f"(before: {detrend.estimate_noise_ppm(lc):.0f} ppm)")
plt.plot(flat_unclipped.time, flat_unclipped.flux, ".", ms=1, color="k", label="kept")
plt.plot(flat_unclipped.time[removed], flat_unclipped.flux[removed], "x", color="C3", label="clipped")
plt.xlabel("Time [BKJD]"); plt.ylabel("Flattened flux"); plt.legend(); plt.show()

## 4. The window-length trade-off
- **Too short** → the smoother has time to dip *into* the transit and follows it: the transit is partly erased.
- **Too long** → stellar wiggles survive and add noise.
- Rule of thumb: **window ≈ 3× the longest transit you want to find.**

We can only *measure* how much depth is lost if we know the true depth, so we use fake data: a spotted star
with a known 4 R⊕ planet injected.

🔮 **Predict:** sketch "measured depth vs window" before running. Where does it flatten out?

In [ ]:
fake = inject.make_synthetic_lightcurve(noise_ppm=50, seed=3)
fake_inj, truth = inject.inject_transit(fake, period=5.0, rp_rearth=4.0, seed=1)
core = detrend.transit_mask(fake.time, 5.0, truth["t0"], truth["duration"], pad=0.6)
print(f"true depth {truth['depth'] * 1e6:.0f} ppm, transit lasts {truth['duration'] * 24:.1f} h")

windows = [0.1, 0.2, 0.3, 0.4, 0.6, 0.75, 1.0, 1.5, 2.0, 3.0]
depths, noises = [], []
for w in windows:
    f = detrend.flatten(fake_inj, window_days=w)
    depths.append(1 - f.flux[core].mean())
    noises.append(detrend.estimate_noise_ppm(f))
    print(f"window {w:4.2f} d = {w / truth['duration']:4.1f} × duration -> depth {depths[-1] * 1e6:5.0f} ppm,"
          f" noise {noises[-1]:4.0f} ppm")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(windows, np.array(depths) / truth["depth"], "o-"); ax[0].axhline(1, color="0.5", ls=":")
ax[0].set_xlabel("window [d]"); ax[0].set_ylabel("measured / true depth"); ax[0].set_xscale("log")
ax[1].plot(windows, noises, "o-", color="C1"); ax[1].set_xlabel("window [d]"); ax[1].set_ylabel("noise [ppm]")
ax[1].set_xscale("log"); plt.tight_layout(); plt.show()

👀 Short windows eat the transit. Long windows keep the depth but let the star's variability back in (noise
rises). Notice that even "good" windows lose ~10–20 % of the depth on this spotted star: inside a window where
the star is brightening or fading, the in-transit points drag the median down. Notebook 03 fixes this (mask the
transits, detrend again).

### ✏️ Your turn 1 🟡
Repeat the window experiment for `method="savgol"` and `method="biweight"` (loop over methods *and* windows).
Which keeps the depth best at short windows? Hippke et al. (2019) tested this on thousands of real stars.
Do you agree with them?

<details><summary>💡 Hint</summary>

Put the loop above inside `for method in ["median", "savgol", "biweight"]:` and pass
`detrend.flatten(fake_inj, method=method, window_days=w)`. Store results in a dict keyed by method and plot
one line per method.
</details>

In [ ]:
# ✏️ YOUR CODE HERE

## 5. BLS intuition: fold at trial periods
BLS asks one question over and over: **"if I fold the light curve at this trial period, is there a box-shaped
dip?"** Before using any algorithm, let's ask it by eye for four trial periods.

🔮 **Predict:** which panel shows a clean dip? What will the 2× panel look like?

In [ ]:
trials = {"0.70 d (wrong)": 0.70, "0.95 d (wrong)": 0.95,
          "0.8375 d (right)": 0.8374907, "1.675 d (2× right)": 2 * 0.8374907}
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharey=True)
for ax, (label, P) in zip(axes.flat, trials.items()):
    plotting.plot_fold(flat, P, 131.5749, ax=ax, window_hours=P * 12, bins=60, title=label)
    ax.set_ylim(-400, 300)
plt.tight_layout(); plt.show()

👀 Wrong periods smear the transits across all phases: flat. The right period stacks them: a dip.
Twice the right period stacks *every other* transit at phase 0, and the rest at ±half a (doubled) period: **two**
dips. That is why BLS periodograms show bumps at 2P, P/2, … (*aliases*).

BLS does this for **tens of thousands** of trial periods, and for each one tries every phase and several
box durations, scoring how much better a box fits than a flat line.

## 6. BLS by hand (a tiny, slow, honest version)

In [ ]:
def my_bls(time, flux, periods, duration, n_phase=100):
    """For each trial period, return the best box SNR over n_phase trial phases."""
    best = np.zeros(len(periods))
    sigma = flux.std()
    for i, P in enumerate(periods):
        phase = (time % P) / P                               # 0..1
        for ph in np.linspace(0, 1, n_phase, endpoint=False):
            d = np.abs(phase - ph); d = np.minimum(d, 1 - d) * P   # days from box centre (wrapping around)
            inside = d < duration / 2                         # points inside the box
            if inside.sum() < 3:
                continue
            depth = flux[~inside].mean() - flux[inside].mean()
            snr = depth / (sigma / np.sqrt(inside.sum()))      # Notebook 00's SNR formula!
            best[i] = max(best[i], snr)
    return best

trial_P = np.linspace(0.5, 3.0, 400)
tic = _time.time()
power_hand = my_bls(flat.time, flat.flux, trial_P, duration=0.08)
print(f"hand BLS took {_time.time() - tic:.1f} s for only {len(trial_P)} periods and ONE duration")
plt.plot(trial_P, power_hand); plt.xlabel("Trial period [d]"); plt.ylabel("best box SNR")
plt.title("Hand-made BLS periodogram"); plt.show()
print("best period:", round(trial_P[power_hand.argmax()], 4), "d")

👀 It works, but it's slow and coarse: 400 periods with a spacing of 0.006 d. Is that fine enough?
A transit lasting $D$ drifts by $\Delta P$ per orbit, so after the whole baseline $T$ (≈ $T/P$ orbits) it has
drifted by $\Delta P \cdot T/P$. That must stay below ~$D$, which needs $\Delta P \lesssim P D / T$ ≈ 0.0006 d here.
Our hand grid (spacing 0.006 d) was ~10× too coarse: notice its best period is off by ~0.001 d. It only worked because the signal is strong.
`search.bls` builds a grid that is uniform in *frequency* with that spacing built in.

## 6b. The real thing: astropy's BoxLeastSquares
Same idea, smarter algorithm (and several durations at once), ~100,000 trial periods in seconds.

In [ ]:
tic = _time.time()
res = search.bls(flat, min_period=0.5, max_period=10)
print(f"astropy BLS: {len(res.periods):,} periods in {_time.time() - tic:.1f} s")
print(res.summary())
plotting.plot_periodogram(res, true_period=0.8374907); plt.show()

## 7. Reading the periodogram
- **Tall peak** = the best period. The dashed lines mark **P/2 and 2P**, where aliases appear.
- **SNR** = depth / depth uncertainty: how significant the *dip* is.
- **SDE** (signal detection efficiency) = how many standard deviations the peak stands above the *rest of the
  periodogram*: how much the period stands out from the noise. A common threshold is SDE ≈ 7–9.
  Week 4 measures what threshold *your* pipeline really needs.

Let's compute SDE ourselves from the periodogram to see there's no magic:

In [ ]:
sde_by_hand = (res.power.max() - res.power.mean()) / res.power.std()
print(f"SDE by hand: {sde_by_hand:.2f}   package: {res.sde:.2f}")

zoom = np.abs(res.periods / res.period - 1) < 0.002          # ±0.2 % around the peak
plt.figure(figsize=(6, 3)); plt.plot(res.periods[zoom], res.power[zoom], ".-")
plt.axvline(0.8374907, color="C2", ls=":", label="published")
plt.xlabel("Period [d]"); plt.ylabel("BLS power"); plt.title("Zoom on the peak"); plt.legend(); plt.show()

👀 The peak has a width: that is the precision BLS gives you on the period (Notebook 03's fit refines it).
Now fold on what BLS found and compare to what physics predicts for the duration of a transit of this star:

In [ ]:
plotting.plot_fold(flat, res.period, res.t0, window_hours=6); plt.show()
dur_exp = utils.expected_transit_duration(res.period, STAR["rstar"], STAR["mstar"])
print(f"published period: 0.8374907 d | yours: {res.period:.5f} d | "
      f"difference: {abs(res.period - 0.8374907) * 86400:.0f} s")
print(f"BLS duration: {res.duration * 24:.2f} h | expected for a central transit of this star: {dur_exp * 24:.2f} h")

👀 The BLS duration only comes from a coarse list of trial durations, so agreement within a factor ~1.5 is
fine. A transit **much longer** than expected hints at a giant star or a blend; much shorter, a grazing orbit.

## 8. Is it a planet? Two vetting tests
Most strong periodic dips in survey data are **not** planets: eclipsing binary stars are far more common. Two
cheap tests catch many of them:

1. **Odd/even depths.** If the true period is 2P (two *different* stars eclipsing each other in turn), odd and
   even "transits" have different depths.
2. **Secondary eclipse.** A star–star binary often shows a second, shallower dip at phase 0.5. A small planet's
   is far too small to see (for Kepler-10 b it's a few ppm).

In [ ]:
n = np.round((flat.time - res.t0) / res.period).astype(int)                # transit number of each point
in_tr = detrend.transit_mask(flat.time, res.period, res.t0, res.duration, pad=0.6)
in_sec = detrend.transit_mask(flat.time, res.period, res.t0 + res.period / 2, res.duration, pad=0.6)

def dip(mask):
    d = 1 - flat.flux[mask].mean()
    e = flat.flux[mask].std() / np.sqrt(mask.sum())
    return d * 1e6, e * 1e6

for name, m in [("odd transits ", in_tr & (n % 2 == 1)), ("even transits", in_tr & (n % 2 == 0)),
                ("phase 0.5    ", in_sec)]:
    d, e = dip(m)
    print(f"{name}: depth = {d:6.1f} ± {e:4.1f} ppm   ({m.sum()} points)")

👀 Odd and even agree within their error bars, and there's nothing at phase 0.5: consistent with a planet.
(Real vetting has many more tests: centroid shifts, transit shape, and comparison with nearby stars. See the
project guide's *Going further* list.)

### ✏️ Your turn 2
1. 🟢 Run BLS **without** detrending (`search.bls(lc, max_period=10)`). What period does it find? Why?
2. 🟢 Change the detrending window to 0.1 d and to 3 d. What happens to the SDE and the depth?
3. 🟡 Run the full chain on **your own target**: load → flatten → clip → BLS → fold → vetting.
   Report P, depth, SNR, SDE and the two vetting results in a table.
4. 🔴 `search.search_multiple(flat, n_planets=3)` masks each planet found and searches again. Kepler-10 has a
   second planet, Kepler-10 c (P ≈ 45.3 d). Can you find it in quarters 3–4? How many transits does it have?
   What could you change to find it?

<details><summary>💡 Hint for 1</summary>

The raw light curve's biggest "signal" is the star's slow variability or the gap between quarters, so BLS
latches onto a long period with a long duration. Detrending first is not optional.
</details>

<details><summary>💡 Hint for 4</summary>

At 45 d there are only ~3–4 transits in ~180 days, and some may fall in gaps. Raise `max_period` to ~60 d
(BLS needs at least 2–3 transits), and consider a longer detrending window: its transit lasts ~7 h.
</details>

In [ ]:
# ✏️ YOUR CODE HERE

### 🚀 Going further (full-timers)
- **TLS:** install `transitleastsquares` and compare its SDE with BLS for a small planet
  (Hippke & Heller 2019 claim ~10 % better sensitivity). Is that true for your star?
- **Multi-planet systems:** TOI-270 (TESS, 3 planets) or Kepler-11 (6 planets!) with `search_multiple`.
- **Your own detrender:** implement a sliding *biweight* (Hippke et al. 2019, Eq. 4) and compare with ours.

## 🧠 Concept check
**Q1.** Your transit lasts 3 hours. Roughly what detrending window should you use?
<details><summary>Answer</summary>About 3× the duration, so ~9 h ≈ 0.4 d. Then check by injecting a fake transit.</details>

**Q2.** The periodogram shows peaks at 2.1, 4.2 and 8.4 d. What are they likely to be?
<details><summary>Answer</summary>One signal and its aliases. The true period is most likely 4.2 d or 2.1 d; fold on each and look (a 2:1 alias at 2P shows dips at phase 0 and 0.5; at P/2 half the "transits" are empty).</details>

**Q3.** Odd transits are 900 ppm deep, even ones 500 ppm (errors ~40 ppm). What do you conclude?
<details><summary>Answer</summary>Probably an eclipsing binary with twice the period: two different stars eclipsing each other in turn.</details>

**Q4.** What does SDE measure that SNR doesn't?
<details><summary>Answer</summary>SDE compares the peak with the whole periodogram: is this <i>period</i> special compared with all the others we tried? SNR only asks whether the dip at that period is significant.</details>

## 📌 Key takeaways
- Detrend to remove slow variability; the window must be longer than the transit and shorter than the wiggles.
- Clip upward outliers only.
- BLS = fold at many trial periods and score a box-shaped dip. Grid spacing must be fine enough.
- Peaks have aliases at P/2, 2P, … Always fold and look.
- Vet: odd/even depths and phase-0.5 dips catch many false positives.

## 📖 Glossary
**trend**: the slow part of the light curve we divide out · **window**: width of the smoothing region ·
**BLS**: Box Least Squares · **periodogram**: detection power vs trial period · **alias**: a false peak at a
simple ratio of the true period · **SDE**: signal detection efficiency · **vetting**: tests that rule out non-planets.

### ✅ Week 2 Checkpoint
- [ ] Kepler-10 b recovered: periodogram + folded plot, committed
- [ ] Your own target: periodogram + fold + a table of (P, depth, SNR, SDE) + vetting results
- [ ] One paragraph: how did window length affect depth? (Include your plot.)

**➡️ Next:** Notebook 03, from "there is a dip" to "this planet is 1.5 ± 0.02 Earth radii".